# Detection Results Viewer

Use this notebook to compare residual baseline, SCC, and CUSUM detection metrics and visualize alarm positions against labelled 2023/2024 operating states.

The purple `Prediction target event` lane is read from the selected method and setting event-summary file. It follows the current test-event definition: auxiliary `Fault`, auxiliary corrective maintenance, status `Stop` with selected IEC categories, and status `Warning` intervals lasting at least 7 days.


In [4]:
from pathlib import Path
import sys
import json
import re

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown, clear_output

try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.read_status import read_status_folder

EXPERIMENTS_DIR = PROJECT_ROOT / "results" / "kelmarsh" / "experiments"
FLAGS_DIR = PROJECT_ROOT / "data" / "interim" / "kelmarsh" / "flags"
FIGURES_DIR = PROJECT_ROOT / "results" / "kelmarsh" / "figures"
TURBINES = [f"Kelmarsh_{i}" for i in range(1, 7)]

DETECTION_METHODS = {
    "residual_baseline": {"label": "Residual baseline", "prefix": "residual_baseline"},
    "scc": {"label": "SCC", "prefix": "scc"},
    "cusum": {"label": "CUSUM", "prefix": "cusum"},
    "bcad": {"label": "BCAD", "prefix": "bcad"},
}

def method_prefix(method):
    return DETECTION_METHODS[method]["prefix"]

def list_experiments():
    if not EXPERIMENTS_DIR.exists():
        return []
    return sorted(p.name for p in EXPERIMENTS_DIR.iterdir() if (p / "metadata.json").exists())

def list_detection_settings(run_id, method):
    run_dir = EXPERIMENTS_DIR / run_id
    prefix = method_prefix(method)
    files = sorted(run_dir.glob(f"{prefix}_performance_*.json"))
    settings = [p.stem.replace(f"{prefix}_performance_", "") for p in files]
    return [setting for setting in settings if "_v" not in setting]

def parse_setting(method, setting):
    if setting is None:
        return {}
    if method == "residual_baseline":
        match = re.fullmatch(r"q(\d+)_c(\d+)", setting)
        if match:
            return {"quantile": int(match.group(1)) / 1000, "min_consecutive": int(match.group(2))}
    if method == "scc":
        match = re.fullmatch(r"abs_meanstd_k([0-9p]+)_c(\d+)", setting)
        if match:
            return {"k": float(match.group(1).replace("p", ".")), "min_consecutive": int(match.group(2))}
    if method == "bcad":
        match = re.fullmatch(r"abs_w(\d+)_q(\d+)_vm([0-9p]+)_c(\d+)", setting)
        if match:
            return {
                "window_size": int(match.group(1)),
                "threshold_quantile": int(match.group(2)) / 1000,
                "variance_multiplier": float(match.group(3).replace("p", ".")),
                "min_consecutive": int(match.group(4)),
            }
    if method == "cusum":
        match = re.fullmatch(r"twosided_(meanstd|std|mad)_k([0-9p]+)_h([0-9p]+)_c(\d+)_(reset|noreset)", setting)
        if match:
            return {
                "scale_type": match.group(1),
                "reference_value": float(match.group(2).replace("p", ".")),
                "decision_threshold": float(match.group(3).replace("p", ".")),
                "min_consecutive": int(match.group(4)),
                "reset": match.group(5),
            }
        match = re.fullmatch(r"twosided_(meanstd|std|mad)_k([0-9p]+)_q(\d+)_c(\d+)_(reset|noreset)", setting)
        if match:
            return {
                "scale_type": match.group(1),
                "reference_value": float(match.group(2).replace("p", ".")),
                "threshold_quantile": int(match.group(3)) / 1000,
                "min_consecutive": int(match.group(4)),
                "reset": match.group(5),
            }
    return {}

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

runs = list_experiments()
if not runs:
    raise FileNotFoundError(f"No experiment metadata found in {EXPERIMENTS_DIR}")

DEFAULT_RUN_ID = runs[-1]
DEFAULT_METHOD = "residual_baseline"
DEFAULT_SETTINGS = list_detection_settings(DEFAULT_RUN_ID, DEFAULT_METHOD)
DEFAULT_SETTING = DEFAULT_SETTINGS[-1] if DEFAULT_SETTINGS else None


## Global Detection Metrics

These metrics are computed for the selected run and setting. For the current `all6` run, they summarize all six turbines, not the turbine selected later in the timeline. Event-level metrics refer to the current prediction target event definition, not all status events/downtime intervals.

In [5]:
def load_selected_result(run_id, method, setting):
    if setting is None:
        raise FileNotFoundError(f"No {method} detection result found for run: {run_id}")
    run_dir = EXPERIMENTS_DIR / run_id
    prefix = method_prefix(method)
    metadata = read_json(run_dir / "metadata.json")
    performance = read_json(run_dir / f"{prefix}_performance_{setting}.json")
    thresholds = pd.read_csv(run_dir / f"{prefix}_thresholds_{setting}.csv")
    return metadata, performance, thresholds

def show_key_metrics(method, run_id, setting):
    clear_output(wait=True)
    metadata, performance, thresholds = load_selected_result(run_id, method, setting)
    setting_info = parse_setting(method, setting)
    display(Markdown(f"### Selected Result: `{DETECTION_METHODS[method]['label']}` / `{run_id}` / `{setting}`"))
    display(pd.DataFrame([{
        "method": DETECTION_METHODS[method]["label"],
        "dataset_id": metadata.get("dataset_id"),
        "turbines": ", ".join(metadata.get("turbines", [])),
        "setting": setting,
        **setting_info,
        "best_epoch": metadata.get("best_epoch"),
        "best_val_loss": metadata.get("best_val_loss"),
    }]))

    event_metrics = performance["event_level"]
    point_metrics = performance["alarm_point_level"]
    episode_metrics = performance["alarm_episode_level"]
    metric_table = pd.DataFrame([
        {
            "level": "event",
            "meaning": "prediction target events detected/missed under current horizon rule",
            "total": event_metrics["total_events"],
            "detected_or_true": event_metrics["detected_events"],
            "missed_or_false": event_metrics["missed_events"],
            "success_rate": event_metrics["detection_rate"],
            "error_rate": event_metrics["miss_rate"],
        },
        {
            "level": "alarm point",
            "meaning": "individual alarm timestamps inside/outside event horizons",
            "total": point_metrics["total_alarm_points"],
            "detected_or_true": point_metrics["true_alarm_points"],
            "missed_or_false": point_metrics["false_alarm_points"],
            "success_rate": point_metrics["true_alarm_points"] / point_metrics["total_alarm_points"] if point_metrics["total_alarm_points"] else 0,
            "error_rate": point_metrics["false_alarm_point_rate"],
        },
        {
            "level": "alarm episode",
            "meaning": "true horizon episodes vs operational false alarm episodes",
            "total": episode_metrics["total_alarm_episodes"],
            "detected_or_true": episode_metrics["true_alarm_episodes"],
            "missed_or_false": episode_metrics.get("operational_false_alarm_episodes", episode_metrics["false_alarm_episodes"]),
            "success_rate": episode_metrics["true_alarm_episodes"] / episode_metrics.get("operational_alarm_opportunities", episode_metrics["total_alarm_episodes"]) if episode_metrics.get("operational_alarm_opportunities", episode_metrics["total_alarm_episodes"]) else 0,
            "error_rate": episode_metrics.get("operational_false_alarm_rate", episode_metrics["false_alarm_episode_rate"]),
            "raw_false_alarm_episode_rate": episode_metrics["false_alarm_episode_rate"],
            "non_operational_alarm_episodes": episode_metrics.get("non_operational_alarm_episodes", 0),
        },
    ])
    display(Markdown("### Key Metrics"))
    display(metric_table.style.format({"success_rate": "{:.2%}", "error_rate": "{:.2%}", "raw_false_alarm_episode_rate": "{:.2%}"}, na_rep=""))

    display(Markdown("### Detection Thresholds / Reference Values"))
    if method == "cusum":
        cols = ["target", "center_type", "scale_type", "center", "scale", "reference_value", "decision_threshold", "validation_samples"]
        display(thresholds[[col for col in cols if col in thresholds.columns]].style.format({
            "center": "{:.4f}",
            "scale": "{:.4f}",
            "reference_value": "{:.3f}",
            "decision_threshold": "{:.3f}",
        }))
    else:
        threshold_view = thresholds.rename(columns={"threshold": "abs_residual_threshold"}).copy()
        threshold_view["lower_error_threshold"] = -threshold_view["abs_residual_threshold"]
        threshold_view["upper_error_threshold"] = threshold_view["abs_residual_threshold"]
        cols = ["target", "quantile", "k", "center", "scale", "lower_error_threshold", "upper_error_threshold", "abs_residual_threshold", "validation_samples"]
        display(threshold_view[[col for col in cols if col in threshold_view.columns]].style.format({
            "quantile": "{:.3f}",
            "k": "{:.3f}",
            "center": "{:.4f}",
            "scale": "{:.4f}",
            "lower_error_threshold": "{:.4f}",
            "upper_error_threshold": "{:.4f}",
            "abs_residual_threshold": "{:.4f}",
        }, na_rep=""))

if HAS_WIDGETS:
    method_dropdown = widgets.Dropdown(
        options=[(v["label"], k) for k, v in DETECTION_METHODS.items()],
        value=DEFAULT_METHOD,
        description="Method",
    )
    run_dropdown = widgets.Dropdown(options=runs, value=DEFAULT_RUN_ID, description="Run")
    setting_dropdown = widgets.Dropdown(options=[], value=None, description="Setting")
    load_metrics_button = widgets.Button(description="Load metrics", button_style="primary")
    metrics_output = widgets.Output()

    def set_setting_options():
        settings = list_detection_settings(run_dropdown.value, method_dropdown.value)
        setting_dropdown.options = settings
        setting_dropdown.value = settings[-1] if settings else None

    def update_settings(*_):
        set_setting_options()
        metrics_output.clear_output(wait=True)

    def on_load_metrics_clicked(_):
        with metrics_output:
            metrics_output.clear_output(wait=True)
            if setting_dropdown.value is None:
                print(f"No result files found for {DETECTION_METHODS[method_dropdown.value]['label']}.")
                return
            show_key_metrics(method_dropdown.value, run_dropdown.value, setting_dropdown.value)

    method_dropdown.observe(update_settings, names="value")
    run_dropdown.observe(update_settings, names="value")
    load_metrics_button.on_click(on_load_metrics_clicked)
    set_setting_options()

    display(widgets.VBox([
        widgets.HBox([method_dropdown, run_dropdown, setting_dropdown, load_metrics_button]),
        metrics_output,
    ]))
else:
    METHOD = DEFAULT_METHOD
    RUN_ID = DEFAULT_RUN_ID
    SETTING = DEFAULT_SETTING
    show_key_metrics(METHOD, RUN_ID, SETTING)


## 2023/2024 Status and Detection Timeline

This plot reads already-built `data/interim/kelmarsh/flags` parquet files, alarm episode CSV files, and the selected setting's event summary. It does not read the raw 2GB SCADA CSV files.

`Prediction target event` includes:
- auxiliary `Fault`
- auxiliary `Maintenance` with `Corrective` or `Corrective - merged`
- status `Stop` with IEC category `Forced outage`, `Out of Electrical Specification`, or `Out of Environmental Specification`
- status `Warning` intervals with duration >= 7 days

`Status event/downtime` remains as a broader background context from `in_event`; it is not the scoring target by itself.

`Operational interval` marks periods that are not inside manual event, status event/downtime, maintenance, communication, curtailment, status records whose IEC category is not `Full Performance`, or `Information/Informational + Out of Environmental Specification` status records with a ±1 hour buffer. Alarm episodes outside target horizons are counted as operational false alarms only when they overlap this operational interval.

In [6]:
def period_bounds(period):
    if period == "2023":
        return pd.Timestamp("2023-01-01"), pd.Timestamp("2024-01-01")
    if period == "2024":
        return pd.Timestamp("2024-01-01"), pd.Timestamp("2025-01-01")
    raise ValueError("Period must be 2023 or 2024")

def load_flag_timeline(turbine_id):
    path = FLAGS_DIR / f"{turbine_id.lower()}_with_flags.parquet"
    if not path.exists():
        raise FileNotFoundError(f"Missing flag file: {path}")
    flag_cols = ["in_maintenance", "in_communication", "in_manual_event", "in_event", "in_curtailment"]
    cols = ["Date and time", "turbine_id"] + flag_cols
    df = pd.read_parquet(path, columns=cols)
    df["Date and time"] = pd.to_datetime(df["Date and time"], errors="coerce")
    return df.dropna(subset=["Date and time"]).sort_values("Date and time")

def flag_intervals(df, flag_col, start, end):
    part = df.loc[(df["Date and time"] >= start) & (df["Date and time"] < end), ["Date and time", flag_col]].copy()
    if part.empty:
        return []
    part[flag_col] = part[flag_col].fillna(False).astype(bool)
    groups = part[flag_col].ne(part[flag_col].shift()).cumsum()
    intervals = []
    for _, group in part.loc[part[flag_col]].groupby(groups):
        s = group["Date and time"].iloc[0]
        e = group["Date and time"].iloc[-1] + pd.Timedelta(minutes=10)
        intervals.append((s, min(e, end)))
    return intervals

def load_non_full_performance_intervals(turbine_id, start, end):
    folder = PROJECT_ROOT / "data" / "raw" / "kelmarsh" / "status" / turbine_id
    if not folder.exists():
        return []
    status = read_status_folder(folder, turbine_id=turbine_id)
    status["Timestamp start"] = pd.to_datetime(status["Timestamp start"], errors="coerce")
    status["Timestamp end"] = pd.to_datetime(status["Timestamp end"], errors="coerce")
    iec = status["IEC category"].astype(str).str.strip().str.casefold()
    status = status[
        status["Timestamp start"].notna()
        & status["Timestamp end"].notna()
        & (status["Timestamp start"] < end)
        & (status["Timestamp end"] >= start)
        & ~iec.eq("full performance")
    ].copy()
    return [(max(row["Timestamp start"], start), min(row["Timestamp end"], end)) for _, row in status.iterrows()]

def load_information_environmental_buffer_intervals(turbine_id, start, end):
    folder = PROJECT_ROOT / "data" / "raw" / "kelmarsh" / "status" / turbine_id
    if not folder.exists():
        return []
    status = read_status_folder(folder, turbine_id=turbine_id)
    status["Timestamp start"] = pd.to_datetime(status["Timestamp start"], errors="coerce")
    status["Timestamp end"] = pd.to_datetime(status["Timestamp end"], errors="coerce")
    status_text = status["Status"].astype(str).str.strip().str.casefold()
    iec = status["IEC category"].astype(str).str.strip().str.casefold()
    status = status[
        status["Timestamp start"].notna()
        & status["Timestamp end"].notna()
        & (status["Timestamp start"] < end)
        & (status["Timestamp end"] >= start)
        & status_text.isin(["information", "informational"])
        & iec.eq("out of environmental specification")
    ].copy()
    buffer = pd.Timedelta(hours=1)
    return [(max(row["Timestamp start"] - buffer, start), min(row["Timestamp end"] + buffer, end)) for _, row in status.iterrows()]

def merge_intervals(intervals, start, end):
    clipped = [(max(s, start), min(e, end)) for s, e in intervals if pd.notna(s) and pd.notna(e) and max(s, start) < min(e, end)]
    if not clipped:
        return []
    clipped = sorted(clipped, key=lambda x: x[0])
    merged = [clipped[0]]
    for s, e in clipped[1:]:
        last_s, last_e = merged[-1]
        if s <= last_e:
            merged[-1] = (last_s, max(last_e, e))
        else:
            merged.append((s, e))
    return merged

def complement_intervals(non_operational_intervals, start, end):
    merged = merge_intervals(non_operational_intervals, start, end)
    operational = []
    cursor = start
    for s, e in merged:
        if cursor < s:
            operational.append((cursor, s))
        cursor = max(cursor, e)
    if cursor < end:
        operational.append((cursor, end))
    return operational

def load_alarm_episodes(run_id, method, setting, turbine_id, start, end):
    prefix = method_prefix(method)
    path = EXPERIMENTS_DIR / run_id / f"{prefix}_alarm_episodes_{setting}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing alarm episode file: {path}")
    episodes = pd.read_csv(path)
    episodes["start_time"] = pd.to_datetime(episodes["start_time"], errors="coerce")
    episodes["end_time"] = pd.to_datetime(episodes["end_time"], errors="coerce")
    return episodes[
        (episodes["turbine_id"] == turbine_id)
        & episodes["start_time"].notna()
        & episodes["end_time"].notna()
        & (episodes["end_time"] >= start)
        & (episodes["start_time"] < end)
    ].copy().sort_values(["target", "start_time"])

def load_target_events(run_id, method, setting, turbine_id, start, end):
    prefix = method_prefix(method)
    path = EXPERIMENTS_DIR / run_id / f"{prefix}_event_summary_{setting}.csv"
    if not path.exists():
        return pd.DataFrame(columns=["event_start", "event_end"])
    events = pd.read_csv(path)
    events["event_start"] = pd.to_datetime(events["event_start"], errors="coerce")
    events["event_end"] = pd.to_datetime(events["event_end"], errors="coerce")
    return events[
        (events["turbine_id"] == turbine_id)
        & events["event_start"].notna()
        & events["event_end"].notna()
        & (events["event_end"] >= start)
        & (events["event_start"] < end)
    ].copy().sort_values("event_start")

def draw_interval_timeline(
    method,
    run_id,
    setting,
    turbine_id,
    period,
    show_false_alarms=True,
    show_non_operational_alarms=True,
    show_true_alarms=True,
):
    clear_output(wait=True)
    start, end = period_bounds(period)
    flags = load_flag_timeline(turbine_id)
    episodes = load_alarm_episodes(run_id, method, setting, turbine_id, start, end)
    target_events = load_target_events(run_id, method, setting, turbine_id, start, end)

    status_lanes = [
        ("Manual event", "in_manual_event", "#d00000"),
        ("Status event/downtime", "in_event", "#ff7b00"),
    ]
    non_operational_intervals = []
    for flag_col in ["in_manual_event", "in_event", "in_maintenance", "in_communication", "in_curtailment"]:
        non_operational_intervals.extend(flag_intervals(flags, flag_col, start, end))
    non_operational_intervals.extend(load_non_full_performance_intervals(turbine_id, start, end))
    non_operational_intervals.extend(load_information_environmental_buffer_intervals(turbine_id, start, end))
    operational_intervals = complement_intervals(non_operational_intervals, start, end)
    targets = list(episodes["target"].dropna().unique())
    lanes = ["Operational interval"] + [name for name, _, _ in status_lanes] + ["Prediction target event"] + [f"Alarm: {target.split('(')[0].strip()}" for target in targets]

    fig_height = max(4.5, 0.48 * len(lanes) + 2.0)
    fig, ax = plt.subplots(figsize=(15, fig_height), facecolor="white")
    ax.set_facecolor("white")
    y_positions = {lane: len(lanes) - 1 - i for i, lane in enumerate(lanes)}
    bar_height = 0.72

    for lane_name, flag_col, color in status_lanes:
        y = y_positions[lane_name]
        for s, e in flag_intervals(flags, flag_col, start, end):
            ax.broken_barh(
                [(mdates.date2num(s), max(mdates.date2num(e) - mdates.date2num(s), 10 / 1440))],
                (y - bar_height / 2, bar_height),
                facecolors=color,
                edgecolors="black",
                linewidth=0.25,
                alpha=0.9,
            )

    target_lane = "Prediction target event"
    y = y_positions[target_lane]
    for _, row in target_events.iterrows():
        s = max(row["event_start"], start)
        e = min(row["event_end"], end)
        ax.broken_barh(
            [(mdates.date2num(s), max(mdates.date2num(e) - mdates.date2num(s), 10 / 1440))],
            (y - bar_height / 2, bar_height),
            facecolors="#7b2cbf",
            edgecolors="black",
            linewidth=0.3,
            alpha=0.9,
        )

    operational_lane = "Operational interval"
    y = y_positions[operational_lane]
    for s, e in operational_intervals:
        ax.broken_barh(
            [(mdates.date2num(s), max(mdates.date2num(e) - mdates.date2num(s), 10 / 1440))],
            (y - bar_height / 2, bar_height),
            facecolors="#2a9d8f",
            edgecolors="black",
            linewidth=0.2,
            alpha=0.75,
        )

    for target in targets:
        lane_name = f"Alarm: {target.split('(')[0].strip()}"
        y = y_positions[lane_name]
        target_eps = episodes.loc[episodes["target"] == target]
        for _, row in target_eps.iterrows():
            s = max(row["start_time"], start)
            e = min(row["end_time"] + pd.Timedelta(minutes=10), end)
            is_true_alarm = bool(row.get("in_fault_horizon", False))
            is_non_operational_alarm = (not is_true_alarm) and bool(row.get("in_non_operational_interval", False))
            is_false_alarm = bool(row.get("is_operational_false_alarm", False))
            if is_true_alarm:
                if not show_true_alarms:
                    continue
                color = "#008000"
            elif is_non_operational_alarm:
                if not show_non_operational_alarms:
                    continue
                color = "#8d99ae"
            elif is_false_alarm:
                if not show_false_alarms:
                    continue
                color = "#111111"
            else:
                if not show_false_alarms:
                    continue
                color = "#111111"
            ax.broken_barh(
                [(mdates.date2num(s), max(mdates.date2num(e) - mdates.date2num(s), 10 / 1440))],
                (y - bar_height / 2, bar_height),
                facecolors=color,
                edgecolors="black",
                linewidth=0.2,
                alpha=0.88,
            )

    ax.set_xlim(mdates.date2num(start), mdates.date2num(end))
    ax.set_yticks([y_positions[lane] for lane in lanes])
    ax.set_yticklabels(lanes)
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.grid(axis="x", color="#d0d0d0", alpha=0.8)
    ax.set_title(f"{turbine_id}: labelled intervals and detection alarms ({period}, {DETECTION_METHODS[method]['label']}, {setting})")
    ax.set_xlabel("Time")
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

    legend_handles = [
        plt.Line2D([0], [0], color="#d00000", lw=6, label="Manual event"),
        plt.Line2D([0], [0], color="#ff7b00", lw=6, label="Status event/downtime"),
        plt.Line2D([0], [0], color="#7b2cbf", lw=6, label="Prediction target event"),
        plt.Line2D([0], [0], color="#2a9d8f", lw=6, label="Operational interval"),
        plt.Line2D([0], [0], color="#008000", lw=6, label="True alarm"),
        plt.Line2D([0], [0], color="#8d99ae", lw=6, label="Ignored non-operational alarm"),
        plt.Line2D([0], [0], color="#111111", lw=6, label="Operational false alarm"),
    ]
    ax.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=4)
    plt.tight_layout()

    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    safe_run_id = str(run_id).replace("/", "_")
    safe_method = str(method).replace("/", "_").replace("\\", "_")
    timeline_png_path = FIGURES_DIR / f"detection_timeline_{safe_method}_{safe_run_id}_{setting}_{turbine_id}_{period}.png"
    timeline_pdf_path = timeline_png_path.with_suffix(".pdf")
    fig.savefig(timeline_png_path, dpi=300, bbox_inches="tight")
    fig.savefig(timeline_pdf_path, bbox_inches="tight")
    print(f"Saved timeline figure to: {timeline_png_path}")
    print(f"Saved PDF figure to: {timeline_pdf_path}")

    plt.show()

    summary = pd.DataFrame([{
        "turbine": turbine_id,
        "period": period,
        "prediction target events": len(target_events),
        "operational intervals": len(operational_intervals),
        "alarm episodes": len(episodes),
        "true alarm episodes": int(episodes.get("in_fault_horizon", pd.Series(dtype=bool)).fillna(False).sum()),
        "ignored non-operational alarm episodes": int((~episodes.get("in_fault_horizon", pd.Series(dtype=bool)).fillna(False) & episodes.get("in_non_operational_interval", pd.Series(dtype=bool)).fillna(False)).sum()) if len(episodes) else 0,
        "operational false alarm episodes": int(episodes.get("is_operational_false_alarm", pd.Series(dtype=bool)).fillna(False).sum()) if len(episodes) else 0,
    }])
    display(summary)

if HAS_WIDGETS:
    timeline_turbine_dropdown = widgets.Dropdown(options=TURBINES, value="Kelmarsh_1", description="Turbine")
    timeline_year_dropdown = widgets.Dropdown(options=["2023", "2024"], value="2023", description="Period")
    show_false_alarm_checkbox = widgets.Checkbox(value=True, description="Show FA")
    show_non_operational_alarm_checkbox = widgets.Checkbox(value=True, description="Show ignored alarms")
    show_true_alarm_checkbox = widgets.Checkbox(value=True, description="Show true alarms")
    draw_timeline_button = widgets.Button(description="Draw timeline", button_style="primary")
    timeline_output = widgets.Output()

    def on_draw_timeline_clicked(_):
        with timeline_output:
            timeline_output.clear_output(wait=True)
            if setting_dropdown.value is None:
                print(f"No result files found for {DETECTION_METHODS[method_dropdown.value]['label']}.")
                return
            draw_interval_timeline(
                method_dropdown.value,
                run_dropdown.value,
                setting_dropdown.value,
                timeline_turbine_dropdown.value,
                timeline_year_dropdown.value,
                show_false_alarm_checkbox.value,
                show_non_operational_alarm_checkbox.value,
                show_true_alarm_checkbox.value,
            )

    draw_timeline_button.on_click(on_draw_timeline_clicked)
    display(widgets.VBox([
        widgets.HBox([timeline_turbine_dropdown, timeline_year_dropdown, draw_timeline_button]),
        widgets.HBox([show_false_alarm_checkbox, show_non_operational_alarm_checkbox, show_true_alarm_checkbox]),
        timeline_output,
    ]))
else:
    TURBINE = "Kelmarsh_1"
    PERIOD = "2023"
    draw_interval_timeline(METHOD, RUN_ID, SETTING, TURBINE, PERIOD)
